# headswap_V2 - test: pre-edit donor expression before T4

**Run order:** Cell 1 (setup, restarts the kernel) -> Cell 2 (upload) -> Cell 3 (run).

Cell 1 detects whether this runtime already has ComfyUI installed:
- **Fresh runtime** -> full setup (clone, deps, ComfyUI, Krea2 weights).
- **Existing runtime** -> skips the slow install, just re-syncs the repo to the latest commit on this branch.

Cell 3 tests the new `pre_edit_donor_expression` step (docs/PIPELINE_STATE.md CHECKPOINT-11/12/13): it measures the **target's** actual expression, edits the **donor** photo to match it (pure Krea2 generation, no mask), then runs T4's existing two-step pass (main pass + face_refine) on the edited donor. The cell displays all three stages: the original donor face, the donor after the expression edit, and T4's final result.


In [ ]:
#@title Cell 1 - Setup (detects an existing runtime; only a fresh one gets the full install)
from pathlib import Path
import subprocess, shutil, os, signal, sys
import importlib.metadata as _im

assert Path("/content").exists(), "Open this notebook in Google Colab."

def _import_torch():
    import torch
    torch.cuda.is_available()  # touch a real attribute to force full init
    return torch

try:
    torch = _import_torch()
except AttributeError as exc:
    # Known Colab base-image hiccup, seen on a brand-new runtime's very
    # first `import torch`: the module partially initializes and a later
    # submodule (torch.fx via torch._export.verifier) is missing, raised
    # as "partially initialized module ... most likely due to a circular
    # import". Reinstalling the SAME pinned version (not upgrading, which
    # could pull a build mismatched with Colab's GPU driver) refreshes
    # whatever got corrupted.
    try:
        _torch_ver = _im.version("torch")
    except _im.PackageNotFoundError:
        _torch_ver = None
    pkg = f"torch=={_torch_ver}" if _torch_ver else "torch"
    print(f"torch import broken on this runtime ({exc}); reinstalling {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps", "--no-cache-dir", pkg],
                   check=True)
    for _m in list(sys.modules):
        if _m == "torch" or _m.startswith("torch."):
            del sys.modules[_m]
    torch = _import_torch()

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run this cell.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"

# Fresh runtime: no ComfyUI on disk yet -> needs the full install below.
# Existing runtime (reconnect / re-run): ComfyUI + weights are already on
# disk -> only re-sync the repo, skip the slow scripts/setup_colab.sh.
FRESH_RUNTIME = not Path("/content/ComfyUI/server.py").exists()
if FRESH_RUNTIME:
    print("-> Fresh runtime detected (no ComfyUI on disk) - running full setup.")
else:
    print("-> Existing runtime detected (ComfyUI already on disk) - skipping "
          "the slow install, just re-syncing the repo.")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
      subprocess.getoutput(f"git -C {REPO} log -1 --pretty=%s"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

if FRESH_RUNTIME:
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("--- stderr ---"); print(r.stderr[-2000:])
        raise SystemExit("setup_colab.sh failed")
else:
    print("ComfyUI + weights already present - skipping scripts/setup_colab.sh")

subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "--no-deps", "numpy==2.4.6"], check=True)

print("\n✓ Setup complete. Restarting kernel (this is expected)...")
print("   When it comes back, run Cell 2.")
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
#@title Cell 2 - Upload YOUR two images
# Upload the BODY first (the photo you want to keep: pose, clothes, background;
# this is also where the DESIRED expression is measured from), then the FACE
# (the donor whose identity you want transferred in -- its expression will be
# edited by Cell 3 to match the body's before the swap).
import os
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "my_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def _grab(role):
    print(f"\n=== Upload the {role} image ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} image uploaded - re-run this cell.")
    name = next(iter(up))
    dest = PAIR / f"{role}.png"
    Image.open(name).convert("RGB").save(dest)
    os.remove(name)
    im = Image.open(dest)
    print(f"saved {role}: {im.size[0]}x{im.size[1]}px")
    if max(im.size) < 500:
        print(f"   note: small source ({im.size[0]}x{im.size[1]}). The pipeline "
              "upscales to 1024 so generated detail survives, but a larger "
              "original will always look sharper.")
    return im

body_im = _grab("body")
face_im = _grab("face")

print("\n--- BODY (kept: pose / clothing / background / DESIRED expression) ---")
display(body_im)
print("--- FACE (donor: identity; expression will be edited to match BODY) ---")
display(face_im)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - Run: donor expression edit (pass 0), then T4's two-step pass
SEED = 46  #@param {type:"integer"}
# First measured attempt (denoise=0.35, ref_boost=2.0, cfg=1.8) produced a
# pixel-identical donor edit -- no visible change. ref_boost anchors fidelity
# to the "person" reference, which here is the SAME photo as the scene, so a
# high value fights the very change the prompt asks for. These new defaults
# lower that anchor, give the sampler more room to move (denoise), and push
# cfg up (Krea2 Turbo needs real cfg headroom before it reliably obeys
# prompt text at all -- see docs/PIPELINE_STATE.md's own cfg 1.0->1.8 note).
# Retune here without another push if this still isn't enough.
PRE_EDIT_DENOISE = 0.55  #@param {type:"number"}
PRE_EDIT_REF_BOOST = 0.5  #@param {type:"number"}
PRE_EDIT_CFG = 3.0  #@param {type:"number"}

import sys, os, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

body_im = Image.open(body_path).convert("RGB")
face_im = Image.open(face_path).convert("RGB")

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "save_debug": False,
    "verbose": False,
    # Pre-step under test (default OFF in the yaml): edit the DONOR's
    # expression to match the TARGET's measured expression before T4's own
    # main pass + face_refine run. See docs/PIPELINE_STATE.md CHECKPOINT-13.
    "pre_edit_donor_expression": True,
    "pre_edit_donor_expression_denoise": float(PRE_EDIT_DENOISE),
    "pre_edit_donor_expression_ref_boost": float(PRE_EDIT_REF_BOOST),
    "pre_edit_donor_expression_cfg": float(PRE_EDIT_CFG),
})

OUT_DIR = REPO / "results" / "pre_edit_expression_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
result = create_pipeline(cfg, runtime=runtime).run(body_im, face_im, out_dir=OUT_DIR)
elapsed = time.perf_counter() - t0

meta = result.meta or {}
pre_edit = meta.get("pre_edit_donor_expression") or {}
route = meta.get("body_route") or {}
edit_mode = meta.get("edit_mode")
route_name = route.get("route")
print(f"\n{elapsed:.0f}s  mode={edit_mode}  route={route_name}  out={result.image.size}")
print(f"pre_edit_donor_expression: applied={pre_edit.get('applied')} reason={pre_edit.get('reason')}")
if pre_edit.get("target_expression"):
    te = pre_edit["target_expression"]
    print(f"  target expression measured: {te.get('label')} "
          f"(smile_ratio={te.get('smile_ratio')} open_ratio={te.get('open_ratio')})")
    print(f"  donor edit knobs: denoise={pre_edit.get('denoise')} "
          f"ref_boost={pre_edit.get('ref_boost')} cfg={pre_edit.get('cfg')} steps={pre_edit.get('steps')}")

pre_edit_face_path = OUT_DIR / "debug_pre_edit_donor_face.png"

display(Markdown("### 1 - Original donor face (uploaded)"))
display(face_im)

if pre_edit.get("applied") and pre_edit_face_path.is_file():
    display(Markdown(
        "### 2 - Donor after the expression edit "
        "(pure Krea2 generation, no mask -- BEFORE T4's two-step pass)"
    ))
    display(Image.open(pre_edit_face_path))
else:
    skip_reason = pre_edit.get("reason")
    display(Markdown(
        f"### 2 - Donor expression edit SKIPPED ({skip_reason}) "
        "-- T4 ran on the original donor face"
    ))

display(Markdown(
    "### 3 - Final result (T4's two-step pass -- main pass + face_refine "
    "-- using the edited donor above)"
))
display(result.image)

final_path = OUT_DIR / "final_result.png"
result.image.save(final_path)
print(f"\nSaved: {final_path}")
if pre_edit_face_path.is_file():
    print(f"Saved: {pre_edit_face_path}")

# Uncomment to download:
# from google.colab import files; files.download(str(final_path))
